# EDA: Пошаговая инструкция для анализа групп признаков

Этот ноутбук содержит единый шаблон для выполнения разведочного анализа данных (EDA) для хакатона ChemAI.
Каждый участник команды:
1. Копирует этот ноутбук в папку `notebooks/` с именем `eda_group<номер>_<фамилия>.ipynb`.
2. Заменяет название группы и список признаков на свои.
3. Выполняет анализ по чек-листу и оформляет вывод в Markdown-ячейке в конце.

**Важно:** после завершения сделайте коммит и Pull Request в ветку `main`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Фиксируем seed для воспроизводимости
SEED = 42
np.random.seed(SEED)

# Настройка графиков
plt.style.use('ggplot')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline


In [ ]:
# Считаем данные (файлы уже должны быть в папке data/)
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

# Переименуем таргеты для удобства
train.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'}, inplace=True)

# Посмотрим первые строки
train.head()


## ⚠️ Замените список признаков на свою группу!
Скопируйте сюда точные имена колонок из train.columns.


In [ ]:
# Пример: Группа 1 (общие молекулярные свойства)
GROUP_NAME = 'Общие молекулярные свойства'
FEATURES = [
    'MolWt', 'ExactMolWt', 'HeavyAtomMolWt', 'NumValenceElectrons',
    'NumRadicalElectrons', 'MolLogP', 'MolMR', 'TPSA', 'LabuteASA',
    'HeavyAtomCount', 'NHOHCount', 'NOCount', 'NumHAcceptors', 'NumHDonors',
    'NumHeteroatoms', 'NumRotatableBonds', 'RingCount',
    'NumAliphaticCarbocycles', 'NumAliphaticHeterocycles', 'NumAliphaticRings',
    'NumAromaticCarbocycles', 'NumAromaticHeterocycles', 'NumAromaticRings',
    'NumSaturatedCarbocycles', 'NumSaturatedHeterocycles', 'NumSaturatedRings',
    'FractionCSP3'
]
TARGETS = ['IC50', 'CC50', 'SI']


## 1. Первичный обзор
Проверим размер, типы данных и примеры значений.


In [ ]:
print(f'Размер train: {train.shape}')
print(f'Количество признаков группы: {len(FEATURES)}')
print(f'Пример данных:\n{train[FEATURES].head(3)}')

# Типы данных
print(f'\nТипы данных:\n{train[FEATURES].dtypes.value_counts()}')

# Описательная статистика
train[FEATURES].describe().round(3)


## 2. Анализ пропусков


In [ ]:
missing = train[FEATURES].isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('✅ Пропуски отсутствуют.')
else:
    print(f'Найдены пропуски в {len(missing)} признаках:')
    print(missing)
    # Визуализация
    sns.heatmap(train[FEATURES].isnull(), cbar=False, cmap='viridis')
    plt.title('Карта пропусков')
    plt.show()


## 3. Анализ распределений
Для каждого признака построим гистограмму и boxplot. Оценим асимметрию и выбросы.


In [ ]:
# Гистограммы
n_cols = 4
n_rows = int(np.ceil(len(FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
axes = axes.flatten()
for i, feat in enumerate(FEATURES):
    ax = axes[i]
    sns.histplot(train[feat], kde=True, ax=ax)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
# Убираем лишние подграфики
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot-диаграммы для выбросов
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows))
axes = axes.flatten()
for i, feat in enumerate(FEATURES):
    ax = axes[i]
    sns.boxplot(y=train[feat], ax=ax)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


## 4. Корреляции
Тепловая карта корреляции признаков группы между собой и с целевыми переменными.


In [ ]:
# Добавляем таргеты к признакам для корреляционной матрицы
corr_cols = FEATURES + TARGETS
corr = train[corr_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # верхний треугольник
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title(f'Корреляция признаков группы {GROUP_NAME}')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Топ-5 признаков по корреляции с каждым таргетом
for target in TARGETS:
    corr_target = corr[target].drop(FEATURES + TARGETS, errors='ignore').sort_values(ascending=False)
    print(f'\nТоп-5 корреляций с {target}:')
    print(corr_target.head(5))


## 5. Поиск проблемных признаков
Найдём константные, почти константные и дублирующиеся признаки.


In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Константные признаки
constant_feats = train[FEATURES].columns[train[FEATURES].std() == 0].tolist()
print(f'Константные (std=0): {constant_feats}' if constant_feats else 'Константных нет.')

# Почти константные (порог дисперсии 0.01)
sel = VarianceThreshold(threshold=0.01)
sel.fit(train[FEATURES])
low_var_feats = [FEATURES[i] for i in np.where(~sel.get_support())[0]]
print(f'Почти константные (var<0.01): {low_var_feats}' if low_var_feats else 'Почти константных нет.')

# Дубликаты корреляций (r > 0.95)
corr_matrix = train[FEATURES].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, idx, upper.loc[col, idx]) for col in upper.columns for idx in upper.index if upper.loc[col, idx] > 0.95]
if high_corr:
    print('Найдены пары с корреляцией >0.95:')
    for pair in high_corr:
        print(f'{pair[0]} – {pair[1]}: {pair[2]:.3f}')
else:
    print('Сильно скоррелированных пар нет.')


## 6. Выводы по группе

**Качество данных:**
- Пропуски: ...
- Выбросы: ...
- Проблемные признаки: ...

**Необходимые преобразования:**
- Масштабирование: (StandardScaler / не требуется) ...
- Импутация: (если есть пропуски) ...
- Удаление признаков: (константные, почти константные, дубликаты) ...

**Наиболее коррелирующие с таргетами признаки:**
- IC50: ...
- CC50: ...
- SI: ...

**Дополнительные замечания:**


# Как отправить результат на проверку (Pull Request)

После того как вы завершили анализ в своей копии ноутбука, выполните шаги ниже.

## 1. Скопируйте ваш ноутбук в репозиторий
- Сначала сохраните копию ноутбука на Google Диск: **Файл → Сохранить копию на Диске**.
- Переименуйте его по шаблону: `eda_group<номер>_<имя>.ipynb`
- Затем в Colab откройте новую ячейку и выполните:

```python
from google.colab import drive
drive.mount('/content/drive')
# Укажите правильный путь к вашему файлу на Диске
!cp "/content/drive/MyDrive/Colab Notebooks/eda_group1_Vadim.ipynb" notebooks/
```

## 2. Создайте новую ветку
```python
!git checkout -b eda-group1-Vadim  # замените на своё имя
```

## 3. Закоммитьте и запушьте ветку
```python
!git add notebooks/eda_group1_Vadim.ipynb  # ваш файл
!git commit -m "EDA group 1: general molecular properties (Vadim)"
!git push origin eda-group1-Vadim
```
При запросе пароля введите **ваш токен GitHub**.

## 4. Создайте Pull Request на GitHub
- Перейдите в репозиторий: https://github.com/NE1nthegame/jazz_team_project
- GitHub предложит создать PR из только что отправленной ветки — нажмите **"Compare & pull request"**.
- Заголовок: например, `EDA Group 1: Общие молекулярные свойства (Вадим)`.
- В описании кратко укажите основные выводы.
- В правой панели **Reviewers** выберите одного из участников (например, Георгия или Андрея).
- Нажмите **Create pull request**.

## 5. Ревью и слияние
- Назначенный ревьюер просмотрит изменения и при необходимости оставит комментарии.
- Когда всё хорошо, ревьюер нажмёт **Approve**, а затем автор (или сам ревьюер) нажмёт **Merge pull request**.
- После слияния ветку можно удалить (кнопка "Delete branch").

## Важные правила
- **НЕ** коммитьте и не пушьте напрямую в `main`.
- Один Pull Request — один ноутбук (одна группа признаков).
- Перед коммитом очистите выводы ячеек: **Runtime → Restart and run all** и убедитесь, что всё работает.
- Проверьте, что в ноутбуке заполнена последняя Markdown-ячейка с выводами.

Если что-то пошло не так — обратитесь в общий чат.
